# Train CellTypist

In [1]:
import anndata as ad
import scanpy as sc
from sklearn.linear_model import LogisticRegression
from skopt import BayesSearchCV
from skopt.space import Real, Categorical
from custom_stopper import CustomStopper
# For saving results on HPC Cluster
import joblib
import pandas as pd
import os

# Load training data
adata = ad.io.read_h5ad('../data/CellTypistDataset/global.h5ad')

# Filter blood data
adata = adata[adata.obs['Organ'] == 'BLD'].copy()
print(adata)


# Preprocessing

# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

# Remove mitochondrial, ribosomal and hemoglobin
adata = adata[:, ~adata.var["mt"]].copy()
adata = adata[:, ~adata.var["ribo"]].copy()
adata = adata[:, ~adata.var["hb"]].copy()

# Doublet Detection
sc.pp.scrublet(adata, batch_key="Donor")


# Normalization

# Saving count data
adata.layers["counts"] = adata.X.copy()

# Normalizing to median total counts
sc.pp.normalize_total(adata)
# Logarithmize the data
sc.pp.log1p(adata)


# Filtering Highly variable genes
print('Before filtering highly variable genes ---')
print(adata)

sc.pp.highly_variable_genes(adata, n_top_genes=10000)

# Apply filter
adata = adata[:, adata.var['highly_variable']].copy()

print('After filtering highly variable genes ---')
print(adata)


# Create train test split

# All Donors: ['621B', '637C', 'A35', 'A36', 'D496', 'D503']
donor_train = ['637C', 'A35', 'A36', 'D503']
donor_test = ['621B', 'D496']

adata_train = adata[
    adata.obs["Donor"].isin(donor_train)
].copy()

adata_test = adata[
    adata.obs["Donor"].isin(donor_test)
].copy()

# Check split
print(adata_train.obs['Donor'].unique())
print(adata_test.obs['Donor'].unique())

# Prepare Data for training
X_train = adata_train.X#.to_df()
gene_names_train = adata_train.var_names
y_train = adata_train.obs['Manually_curated_celltype']

X_test = adata_test.X#.to_df()
gene_names_train = adata_train.var_names
y_test = adata_test.obs['Manually_curated_celltype']

AnnData object with n_obs × n_vars = 27620 × 36601
    obs: 'Organ', 'Donor', 'Chemistry', 'Cell_category', 'Predicted_labels_CellTypist', 'Majority_voting_CellTypist', 'Majority_voting_CellTypist_high', 'Manually_curated_celltype'
    obsm: 'X_umap'
Before filtering highly variable genes ---
AnnData object with n_obs × n_vars = 27620 × 36473
    obs: 'Organ', 'Donor', 'Chemistry', 'Cell_category', 'Predicted_labels_CellTypist', 'Majority_voting_CellTypist', 'Majority_voting_CellTypist_high', 'Manually_curated_celltype', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet'
    var: 'mt', 'ribo', 'hb'

In [4]:
import celltypist

model = celltypist.train(X_train, y_train, gene_names_train)

print(model.score(X_test, y_test))

🍳 Preparing data before training
👀 The input training data is processed as an array-like object


ValueError: 🛑 Invalid expression matrix, expect log1p normalized expression to 10000 counts per cell

In [6]:
import celltypist

model = celltypist.train(
    X=adata_train,                          # Das AnnData-Objekt für das Training
    labels='Manually_curated_celltype',     # Die Spalte in adata_train.obs
    genes=None,                             # None bedeutet: alle Gene in adata_train.var_names nutzen
    check_expression=False,                  # Prüft, ob die Daten log-normalisiert sind
)

# Optional: Modell für später speichern
# model_ct.write('mein_custom_celltypist_model.pkl')

print("Training abgeschlossen. Starte Vorhersage auf Testdaten...")

# 2. Vorhersage (Inferenz) auf den Testdaten ausführen
# celltypist.annotate gibt ein AnnotationResult-Objekt zurück
predictions = celltypist.annotate(
    X=adata_test, 
    model=model,           # Unser frisch trainiertes Modell
    majority_voting=False     # Auf True setzen, wenn du die Labels über Cluster glätten willst
)

# 3. Score (Accuracy) berechnen
# Die vorhergesagten Labels liegen in predictions.predicted_labels
y_pred = predictions.predicted_labels['predicted_labels']
y_true = adata_test.obs['Manually_curated_celltype']

# Genauigkeit berechnen
accuracy = accuracy_score(y_true, y_pred)
print(f"CellTypist Test Accuracy: {accuracy:.4f}")

🍳 Preparing data before training
✂️ 304 non-expressed genes are filtered out
🔬 Input data has 18325 cells and 9696 genes
⚖️ Scaling input data
🏋️ Training data using logistic regression


TypeError: LogisticRegression.__init__() got an unexpected keyword argument 'multi_class'